In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

PAGEINDEX_API_KEY = os.getenv("PAGE_INDEX_API_KEY")
GEMINI_API_KEY = os.getenv("GOOGLE_API_KEY")

In [2]:
from pageindex import PageIndexClient
from google import genai

pageindex_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

1. Upload the file and request tree creation

In [13]:
pdf_path = "Data/PdfFiles/ChocoKari_Info.pdf"
result = pageindex_client.submit_document(pdf_path)
doc_id = result["doc_id"]
print(f"File uploaded with document id: {doc_id}")

File uploaded with document id: pi-cmuwqllpp00000bpezl1hklee


2. Wait for tree creation

In [4]:
import time

while True:
    document = pageindex_client.get_document(doc_id)
    document_status = document.get("status")
    if (document_status == "completed"):
        print("File processed successfully.")
        break
    elif (document_status == "failed"):
        print("File processing failed. Please retry.")
        break
    time.sleep(5)

File processed successfully.


3. Fetch full tree

In [5]:
import json

tree_result = pageindex_client.get_tree(doc_id, node_summary=True)
tree = tree_result.get("result",[])
print(f"Total top level sections: {len(tree)}")
print("RAW TREE NODE :")
print(json.dumps(tree[0] if tree else {}, indent=4))

Total top level sections: 17
RAW TREE NODE :
{
    "title": "1. BRAND & BUSINESS OVERVIEW",
    "node_id": "0000",
    "start_index": 1,
    "end_index": 1,
    "summary": "# 1. BRAND & BUSINESS OVERVIEW\n\nBrand Name: ChocoKari\n\nStory: Born in a home kitchen, perfected in every batch. Each piece is hand-tempered, blended with the finest inclusions, and finished to a silky shine. Small-batch chocolate made for life's sweetest moments.\n\nTagline: Every Bite, A Story to Savor\n\nBusiness Type: Handcrafted artisan chocolate, made in small batches in India.\n\nMission: Premium handcrafted chocolates using the world's finest ingredients.\n\nLocation and Delivery: India \u2014 we deliver pan-India.\n",
    "text": "# 1. BRAND & BUSINESS OVERVIEW\n\nBrand Name: ChocoKari\n\nStory: Born in a home kitchen, perfected in every batch. Each piece is hand-tempered, blended with the finest inclusions, and finished to a silky shine. Small-batch chocolate made for life's sweetest moments.\n\nTagline

4. Print full tree structure

In [6]:
def print_tree(sub_tree, indent):
    for node in sub_tree:
        prefix = " " * indent + ("└─" if indent > 0 else "")
        print(f"{prefix} [{node["node_id"]}] {node["title"]} (p.{node["start_index"]}-{node["end_index"]})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent+1)

print("FULL TREE STRUCTURE :")
print_tree(tree,0)

FULL TREE STRUCTURE :
 [0000] 1. BRAND & BUSINESS OVERVIEW (p.1-1)
 [0001] 2. COLLECTIONS (p.1-1)
 [0002] 3. FLAVOURS (p.1-3)
 [0003] 4. CHOCOLATE INTENSITY (p.3-3)
 [0004] 5. STICKERS & PERSONALIZATION (p.3-3)
 [0005] 6. BOX SIZES (p.3-3)
 [0006] 7. CUSTOM CHOCOLATE BUILDER (p.3-4)
 [0007] 8. CORPORATE GIFTING & BULK ORDERS (p.4-4)
 [0008] 9. CHOCOLATE MATCHMAKER (p.4-4)
 [0009] 10. SHIPPING & DELIVERY (p.4-4)
 [0010] 11. PAYMENT (p.4-4)
 [0011] 12. CANCELLATION & REFUNDS (p.4-5)
 [0012] 13. CUSTOMER ACCOUNT (p.5-5)
 [0013] 14. FOOD & INGREDIENT INFORMATION (p.5-5)
 [0014] 15. WEBSITE FEATURES (p.5-5)
 [0015] 16. WEBSITE PAGES (p.5-5)
 [0016] 17. CONTACT (p.5-5)


5. Count total number of nodes

In [7]:
def count_nodes(sub_tree):
    total = len(sub_tree)
    for node in sub_tree:
        if node.get("nodes"):
            total += count_nodes(node["nodes"])
    return total

print("TOTAL NUMBER OF NODES IN THE TREE :")
print(count_nodes(tree))

TOTAL NUMBER OF NODES IN THE TREE :
17


6. LLM tree search

In [ ]:
from google.genai import types

def llm_tree_search(query : str, tree : list, model : str) -> dict:
    def compress_tree(nodes):
        compressed = []
        for node in nodes:
            new_node = {
                "title" : node["title"],
                "node_id" : node["node_id"],
                "summary" : node["summary"]
            }
            if node.get("nodes"):
                new_node["nodes"] = compress_tree(node["nodes"])
            compressed.append(new_node)
        return compressed

    compressed_tree = compress_tree(tree)

    prompt = f"""
    You are given a query and a document's tree structure (like a Table of Contents).
    Your task : Identify which node ids most likely contain the answer to the query.
    Query : {query}
    Document Tree : {json.dumps(compressed_tree, indent=4)}
    Reply ONLY in this exact JSON format:
    {{
        "thinking" : "<your step-by-step reasoning>",
        "node_list" : ["node_id1","node_id2"]
    }}
    """

    response = gemini_client.models.generate_content(
        model=model,
        contents=prompt,
        config=types.GenerateContentConfig(
            response_mime_type="application/json"
        )
    )

    return json.loads(response.text)

7. Full end to end RAG pipeline

In [9]:
def find_node_by_id(tree : list, node_list : list) -> list:
    target_node = []
    for node in tree:
        if node["node_id"] in node_list:
            target_node.append(node)
        if node.get("nodes"):
            target_node.extend(find_node_by_id(node["nodes"],node_list))
    return target_node

In [10]:
def generate_answer(query : str, nodes : list, model : str) -> str:
    if not nodes:
        return "No relevant sections found in the document."
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"Section : {node["title"]} \n"
            f"{node.get("text","Content not available.")}"
        )
    context = "\n\n --- \n\n".join(context_parts)
    prompt = f"""
    You are an expert in document analysis.
    Answer the question using ONLY the provided context.
    Question : {query}
    Context : {context}
    Answer : 
    """
    response = gemini_client.models.generate_content(
        model=model,
        contents=prompt
    )
    return response.text

In [ ]:
def vectorless_rag(query : str, tree : list, model : str) -> str:
    search_result = llm_tree_search(query, tree, model)
    node_ids = search_result.get("node_list",[])
    node_list = find_node_by_id(tree, node_ids)
    response = generate_answer(query, node_list, model)
    return response

In [13]:
response = vectorless_rag("What collections and flavours are available at ChocoKari?", tree, "gemini-3.5-flash-lite")
response

"Based on the provided context, the available collections and flavours at ChocoKari are:\n\n**Collections:**\n1. **Classic** (Timeless favourites for everyday indulgence and casual gifting)\n2. **Signature** (Chef's carefully crafted combinations for thoughtful gifting and special moments)\n3. **Royale** (Premium single-origin luxury chocolates for celebrations and premium gifting)\n\n**Flavours:**\n1. **Roasted Almond** (Roasted almonds in milk and dark chocolate)\n2. **Hazelnut** (Hazelnuts and nutella enrobed in milk and dark chocolate)\n3. **Kunafa** (Middle Eastern kunafa with pistachio)\n4. **Biscoff** (Dark chocolate filled with creamy Biscoff spread)\n5. **Royal Rose** (White chocolate infused with rose petals and dry fruits)\n6. **Cranberry** (Tangy cranberries wrapped in white chocolate)\n7. **Paan** (Betel-leaf inspired chocolate with gulkand)\n8. **Oreo Delight** (Crushed Oreos in creamy milk chocolate)\n9. **Fresh Mint** (Cool peppermint in a dark chocolate shell)\n10. **B